# Evaluate the HelpSteer2 Fixed Lambda Sweep

This notebook evaluates the fixed five-objective HelpSteer2 adapter merges stored in:

`results/helpsteer2_adapter_merge_generations.csv`

It adds lightweight heuristic proxy scores for helpfulness, correctness, coherence, complexity, and verbosity, then summarizes the tested merge coefficients under several example preference vectors.

**Important:** These proxies are placeholder engineering checks. They are not reward-model scores, factual correctness measurements, or human HelpSteer2 labels. This notebook does not train or merge adapters, compute a relationship matrix, or run preference-aware coefficient correction.

## Clone or update the repository

The following cell starts from `/content`. It updates an existing valid repository or clones a fresh copy, avoiding nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## Inspect the repository

Confirm that the evaluation script and `results/` folder are available.

In [ ]:
!pwd
!ls
!ls scripts
!ls results || echo "No results folder found yet."

## Install the preview dependency

The evaluator itself uses standard Python libraries. Pandas is installed only for convenient CSV previews.

In [ ]:
!pip install -q pandas

## Check the required input file

The evaluation requires `results/helpsteer2_adapter_merge_generations.csv`, which is created by the fixed HelpSteer2 adapter merge evaluation.

If the file is missing, run Notebook 09 or execute:

```bash
python scripts/evaluate_helpsteer2_adapter_merges.py
```

In [ ]:
from pathlib import Path

input_path = Path("results/helpsteer2_adapter_merge_generations.csv")

if not input_path.is_file():
    raise FileNotFoundError(
        f"Missing input file: {input_path}. "
        "Run Notebook 09 or scripts/evaluate_helpsteer2_adapter_merges.py first."
    )

print(f"Input file found: {input_path}")

## Preview the fixed-merge generations

This quick preview confirms that the input contains merge names, five lambda columns, prompts, and generated responses.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 100)
generations_df = pd.read_csv(input_path)
print(f"Rows: {len(generations_df)}")
generations_df.head()

## Run the HelpSteer2 proxy evaluation

The script scores each generated response, aggregates results by `merge_name` and the complete lambda vector, and calculates utilities for four example preference vectors.

In [ ]:
!python scripts/evaluate_helpsteer2_lambda_sweep.py

## Preview the scored generations

`results/helpsteer2_adapter_merge_scored_generations.csv` contains one row per generated response with the five proxy scores, response length, and an empty-response flag.

In [ ]:
scored_path = Path("results/helpsteer2_adapter_merge_scored_generations.csv")
scored_df = pd.read_csv(scored_path)

print(f"Rows: {len(scored_df)}")
display(scored_df.head())

## Preview the lambda-sweep summary

`results/helpsteer2_lambda_sweep_summary.csv` contains one row per tested merge. It reports mean attribute proxies and utilities for these example preferences:

- `balanced`: `[0.2, 0.2, 0.2, 0.2, 0.2]`
- `quality_focused`: `[0.3, 0.3, 0.3, 0.05, 0.05]`
- `detailed_answer`: `[0.25, 0.25, 0.2, 0.15, 0.15]`
- `helpfulness_focused`: `[0.6, 0.1, 0.1, 0.1, 0.1]`

Each utility is the weighted sum of the five mean proxy scores. A higher value only means that a tested merge scored higher under these provisional heuristics.

In [ ]:
summary_path = Path("results/helpsteer2_lambda_sweep_summary.csv")
summary_df = pd.read_csv(summary_path)

print(f"Merge summaries: {len(summary_df)}")
display(summary_df)

## Interpretation limits

The proxy scores use simple textual cues. For example, the correctness proxy rewards explanatory and calibrated wording but cannot verify factual truth. The verbosity proxy is primarily normalized response length. These results are useful for checking the experiment pipeline, not for drawing final conclusions about model quality.

## Git safety check

It is acceptable to commit the two small CSV result files. Do not commit `adapters/`, ZIP backups, `.safetensors`, `.bin`, checkpoints, or other model files.

In [ ]:
!git status